### LECL支持在运行的时候对链进行配置
---
* 动态改写模型的温度
* 动态切换提示词

#### 动态改写模型的温度

In [53]:
from langchain_core.runnables import ConfigurableField
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
).configurable_fields(
    temperature=ConfigurableField(
        id="llm_temperature",
        name="LLM Temperature",
        description="The temperature of the LLM",
    )
)

model.invoke("随意挑选一个随机数，输出为一个整数")


AIMessage(content='\n\n42', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 61, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 69, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 61}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.8-27B', 'system_fingerprint': '', 'id': '01a1a96242358f378a85a3610504a907', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a962-3d72-7431-898c-afdfff2f791a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 72, 'total_tokens': 133, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 69}})

In [54]:

# 在运行时改写温度
model.with_config(configurable={"llm_temperature": 1}).invoke("随意挑选一个随机数，输出为一个整数")

AIMessage(content='\n\n42', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 61, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 47, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 61}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.8-27B', 'system_fingerprint': '', 'id': '01a1a9624de32119bdc9ed851fa9e8cc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1a962-4965-7352-bfc2-128216e11424-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 50, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 47}})

#### 链的提示词动态切换

In [ ]:
from langsmith import Client
from langchain_core.runnables import RunnableLambda, RunnableConfig

# 初始化 Hub 客户端
hub_client = Client()

# 定义替代 HubRunnable 的函数
def dynamic_hub_invoke(inputs: dict, config: RunnableConfig):
    # 1. 获取动态配置的 commit，默认为 "rlm/rag-prompt"
    commit = config.get("configurable", {}).get("hub_commit", "rlm/rag-prompt")
    
    # 2. 拉取 Prompt (必须添加 dangerously_pull_public_prompt=True 参数)
    prompt = hub_client.pull_prompt(
        commit, 
        dangerously_pull_public_prompt=True
    )
    
    # 3. 执行并返回
    return prompt.invoke(inputs)

# 包装成 Runnable
prompt = RunnableLambda(dynamic_hub_invoke)

# ================= 测试调用 =================

# 场景 A：使用默认 Prompt
result_default = prompt.invoke({"question": "foo", "context": "bar"})
print("--- 默认版本 ---")
print(result_default)

# 场景 B：动态指定另一个 Commit (原代码 configureable_fields 的核心功能)
# 在运行时切换提示词
result_custom = prompt.invoke(
    {"question": "foo", "context": "bar"},
    config={
        "configurable": {
            "hub_commit": "rlm/rag-prompt-llama" # 生产环境中建议替换为具体的 commit hash
        }
    }
)
print("\n--- 动态指定版本 ---")
print(result_custom)

--- 默认版本 ---
messages=[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: foo \nContext: bar \nAnswer:", additional_kwargs={}, response_metadata={})]

--- 动态指定版本 ---
messages=[HumanMessage(content="[INST]<<SYS>> You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.<</SYS>> \nQuestion: foo \nContext: bar \nAnswer: [/INST]", additional_kwargs={}, response_metadata={})]


In [ ]:
# 💡 导入基础的 PromptTemplate
from langchain_core.prompts import PromptTemplate 
from langchain_core.runnables import ConfigurableField

# 💡 使用 PromptTemplate 实例化，它拥有顶级的 template 属性
base_prompt = PromptTemplate(
    template="你是一个问答助手。请根据以下上下文回答问题。\n上下文: {context}\n问题: {question}\n回答:",
    input_variables=["context", "question"]
)

# 2. 演示 configurable_fields 的用法 
# 现在 'template' 是一个合法且存在的配置键了
configurable_prompt = base_prompt.configurable_fields(
    template=ConfigurableField(
        id="prompt_template_id", 
        name="Prompt Template",
        description="The prompt template to use for the chain"
    )
)

# 3. 正常调用 (使用默认的 template)
print("--- 默认调用 ---")
result1 = configurable_prompt.invoke({"question": "foo", "context": "bar"})
print(result1)

# 在运行时切换提示词
# 4. 动态覆盖配置调用 (演示 configurable 的威力)
print("\n--- 动态覆盖配置调用 ---")
new_template = "请用一句话简短回答：{question} (背景信息: {context})"

result2 = configurable_prompt.invoke(
    {"question": "foo", "context": "bar"},
    config={"configurable": {"prompt_template_id": new_template}}
)
print(result2)

--- 默认调用 ---
text='你是一个问答助手。请根据以下上下文回答问题。\n上下文: bar\n问题: foo\n回答:'

--- 动态覆盖配置调用 ---
text='请用一句话简短回答：foo (背景信息: bar)'
